In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine
from sklearn.preprocessing import normalize
from scipy.spatial.distance import cosine

def build_cart_ensemble(X_train, y_train, n_estimators=100, max_features='sqrt'):
    """Train Bagging ensemble of CARTs"""
    bagger = BaggingClassifier(
        base_estimator=DecisionTreeClassifier(),
        n_estimators=n_estimators,
        max_features=max_features,
        bootstrap=True
    )
    bagger.fit(X_train, y_train)
    return bagger.estimators_

def evaluate_cart_accuracy(cart, test_sets):
    """Evaluate CART over 3 different test sets and return average accuracy"""
    accuracies = [accuracy_score(y, cart.predict(X)) for X, y in test_sets]
    return np.mean(accuracies)

def compute_feature_vector(tree):
    """Return a binary vector indicating which features were used in the tree"""
    n_features = tree.n_features_in_
    used_features = np.zeros(n_features)
    features = tree.tree_.feature
    for f in features:
        if f != -2:  # -2 is the constant used in sklearn for leaf node
            used_features[f] = 1
    return used_features

def compute_correlation_matrix(trees):
    """Compute cosine similarity between all pairs of feature vectors"""
    vectors = np.array([compute_feature_vector(tree) for tree in trees])
    vectors = normalize(vectors)
    n = len(trees)
    correlation_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            similarity = 1 - cosine(vectors[i], vectors[j])
            correlation_matrix[i, j] = correlation_matrix[j, i] = similarity
    return correlation_matrix

def improved_random_forest(X, y, n_estimators=50, oversample_factor=0.2, correlation_threshold=0.9):
    # Step 1: Split into 4 parts
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.6, stratify=y)
    X_test1, X_test2, y_test1, y_test2 = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)
    X_test3, X_test4, y_test3, y_test4 = train_test_split(X_test2, y_test2, test_size=0.5, stratify=y_test2)
    test_sets = [(X_test1, y_test1), (X_test3, y_test3), (X_test4, y_test4)]

    total_estimators = int(n_estimators * (1 + oversample_factor))
    carts = build_cart_ensemble(X_train, y_train, n_estimators=total_estimators)

    # Step 2: Evaluate each CART
    avg_accs = np.array([evaluate_cart_accuracy(cart, test_sets) for cart in carts])

    # Step 3: Compute correlation matrix
    corr_matrix = compute_correlation_matrix(carts)

    # Step 4: Filter trees
    deletable = set()
    for i in range(len(carts)):
        for j in range(i + 1, len(carts)):
            if corr_matrix[i, j] > correlation_threshold:
                worse = i if avg_accs[i] < avg_accs[j] else j
                deletable.add(worse)

    # Step 5: Keep top N trees not in deletable
    candidates = [(i, acc) for i, acc in enumerate(avg_accs) if i not in deletable]
    candidates = sorted(candidates, key=lambda x: -x[1])
    selected_indices = [idx for idx, _ in candidates[:n_estimators]]

    selected_carts = [carts[i] for i in selected_indices]
    return selected_carts

# ========== Example Usage ==========
data = load_wine()
X, y = data.data, data.target

selected_trees = improved_random_forest(X, y, n_estimators=50, oversample_factor=0.3, correlation_threshold=0.85)

# Evaluate final ensemble
def majority_vote(trees, X):
    from scipy.stats import mode
    predictions = np.array([tree.predict(X) for tree in trees])
    return mode(predictions, axis=0).mode[0]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
y_pred = majority_vote(selected_trees, X_test)

print("Final Accuracy:", accuracy_score(y_test, y_pred))